---
layout: post
title:  Multi-Token Prediction on OJAI
date:   2026-05-15
categories: [AI, ROCm]
toc: true
mermaid: true
maths: true
typora-root-url: ~/Github/ojitha.github.io
typora-copy-images-to: ../../blog/assets/images/${filename}
---

{% include video-summary.html
   id=""
   content="" %}

<!--more-->

* TOC
{:toc}

---

# Introduction
This course about integrating and deploying Claude through Amazon Bedrock[^1]. Accroding to the Claude[^2], there are three models from the Claude:

![Cluade Models](https://academy.claude.com/assets/media/d87ca18bfca0fadd839a95aa8eecb912a4b3cf1dbb1400081f2324be879509a9.png)
Image from [Claude Academy](https://academy.claude.com/assets/media/d87ca18bfca0fadd839a95aa8eecb912a4b3cf1dbb1400081f2324be879509a9.png)

I am using Amazon Bedroc in `ap-southeast-2` which is the closest to the Sydney. It is important to find the available models and the Model IDs in the region. Currently my system has the following versions:

In [4]:
%%bash
aws bedrock list-foundation-models --by-provider anthropic --query "modelSummaries[*].modelId" --output table

-----------------------------------------------
|            ListFoundationModels             |
+---------------------------------------------+
|  anthropic.claude-haiku-4-5-20251001-v1:0   |
|  anthropic.claude-fable-5                   |
|  anthropic.claude-sonnet-4-6                |
|  anthropic.claude-opus-4-6-v1               |
|  anthropic.claude-opus-5                    |
|  anthropic.claude-opus-4-8                  |
|  anthropic.claude-opus-4-7                  |
|  anthropic.claude-sonnet-4-5-20250929-v1:0  |
|  anthropic.claude-fable-5-1                 |
|  anthropic.claude-sonnet-5                  |
|  anthropic.claude-opus-4-5-20251101-v1:0    |
|  anthropic.claude-sonnet-4-20250514-v1:0    |
+---------------------------------------------+


The command `aws bedrock list-foundation-models --by-provider anthropic --query "modelSummaries[*].modelId" --output table` lists all available Anthropic foundation models in Amazon Bedrock.

**Breakdown:**

| Part | Description |
|------|-------------|
| `aws bedrock` | AWS CLI service for Amazon Bedrock |
| `list-foundation-models` | Operation to retrieve available foundation models |
| `--by-provider anthropic` | Filters results to only show models from Anthropic |
| `--query "modelSummaries[*].modelId"` | JMESPath query to extract only the `modelId` field from each model summary |
| `--output table` | Formats output as an ASCII table for readability |

> **Choose Sonnet** when you need balance. Most applications benefit from Sonnet's combination of intelligence, speed, and reasonable cost.
{.ok}

Essential component to connect to the Bedrock Model:

1. Bedrock runtime client
2. Model ID
3. Prompt message

You can create client conntecting the Bedrock runtime:

In [11]:
import boto3

client = boto3.client('bedrock-runtime', region_name='ap-southeast-2')

![User Inference profile](https://academy.claude.com/assets/media/4789ffaf0596fa27ae75b1d8b18808aebeb282677f9b7eac625a369f601208ac.png)
Image from [Claude Academy](https://academy.claude.com/assets/media/4789ffaf0596fa27ae75b1d8b18808aebeb282677f9b7eac625a369f601208ac.png)

> Inference profile automatically route the request to the region where your choosen model is available.
{.ok}



As per above you can use the default Opus, Sonnet, or  

In [45]:
user_message = {
    "role": "user",
    "content": [
        {"text": "What is the capital of Sri Lanka?"}
    ]
}

response = client.converse(
    modelId='au.anthropic.claude-opus-4-8',
    messages=[user_message],
)

In [46]:
print(response["output"]["message"]["content"][0]["text"])

Sri Lanka has two capitals:

1. **Sri Jayawardenepura Kotte** – This is the official (administrative) capital, where the parliament and legislative functions are located. It's often considered the "official" capital.

2. **Colombo** – This is the commercial capital and largest city. It serves as the executive and judicial center and is frequently referred to as the capital in casual contexts.

Sri Jayawardenepura Kotte is actually a suburb of the larger Colombo metropolitan area, which is why there's often some confusion. The capital was officially moved from Colombo to Sri Jayawardenepura Kotte in 1982.


## Multi-turn conversation
Both Bedrock runtime and Claude model don't store any messages. Therefore, for a conversation where you need to store the history. You can  mannualy or programmatically maintain the history of all the messages in the follow up prompt: this is call ***context***.

> Conversation should follow the `user → assistant → user → assistant` pattern.
{.warn}

## System Prompts



[^1]: [Claude with Amazon Bedrock · Claude Academy](https://academy.claude.com/courses/claude-with-amazon-bedrock){:target="_blank" rel="noopener noreferrer"}

[^2]: [Overview of Claude Models · Claude with Amazon Bedrock · Claude Academy](https://academy.claude.com/courses/claude-with-amazon-bedrock/overview-of-claude-models){:target="_blank" rel="noopener noreferrer"}